### Install New Libraries

In [118]:
#!pip install ddgs trafilatura -q # -q without any logs

### Load APIs and Libraries

In [167]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from ddgs import DDGS
import trafilatura
from IPython.display import display, Markdown

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is Missing")

client = OpenAI()
MODEL="gpt-4.1-mini"

In [146]:
def search_web(query):
    """ Search the web using DuckDuckGo browser. Return 3 results."""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=10)
    print(f"\u2705 got results")
    return json.dumps(results, indent=2)

In [147]:
# Stand alone
url = "https://en.wikipedia.org/wiki/Artificial_intelligence_in_healthcare"
url1 = "https://www.linkedin.com/posts/kazi-jannatun-nayeem_healthcare2030-digitalhealth-artificialintelligence-activity-7481350569320734721-yqoJ"
url2 = "https://www.weforum.org/stories/2025/08/ai-transforming-global-health/"
downloaded = trafilatura.fetch_url(url2)
#print(downloaded)
print(f"\n=====================\n")
if downloaded:
    content = trafilatura.extract(
        downloaded,
        # include_links = True,
        # include_tables=True
    )
    print(content)

In [148]:
def fetch_url(url):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 Got text: {len(text)} chars")
            return text
    print(f"\u274c Failed to fetch or extract test fron {url}.")
    return f"Could not extract the text from {url}. try a different source"

In [149]:
search_web("Apples in Madagascar")

✅ got results


'[\n  {\n    "title": "Now I can eat apples! - YouTube",\n    "href": "https://www.youtube.com/watch?v=IxJUN6MCnKM",\n    "body": "Sep 7, 2021 ... Now I can eat apples! @iamnotstevebuscemi130 likes5.1K views4 years ago more. Subscribe. Comments. 12. Comment."\n  },\n  {\n    "title": "Apples Price in Madagascar, July 2026 - Selina Wamucii",\n    "href": "https://www.selinawamucii.com/insights/prices/madagascar/apples/",\n    "body": "As of July 2026, the apples price in Madagascar available is US$ 1.08 per kg (4,796 MGA). Madagascar exported apples worth US$434 in 2020 ..."\n  },\n  {\n    "title": "Now I can eat apples! | Madagascar 3: Europe\'s Most Wanted - Yarn",\n    "href": "https://www.getyarn.io/yarn-clip/8fc25905-2e82-43bd-bab0-226c17be8b89",\n    "body": "Madagascar 3: Europe\'s Most Wanted clip with quote Now I can eat apples! Yarn is the best search for video clips by quote. Find the exact moment in a TV ..."\n  },\n  {\n    "title": "Baked Apples with Madagascar Vanilla Bu

In [150]:
fetch_url("https://www.weforum.org/stories/2025/08/ai-transforming-global-health/")

❌ Failed to fetch or extract test fron https://www.weforum.org/stories/2025/08/ai-transforming-global-health/.


'Could not extract the text from https://www.weforum.org/stories/2025/08/ai-transforming-global-health/. try a different source'

### Step 2: Describe as LLM Tool calling

In [151]:
tools = []

In [153]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using DuckDuckGo browser. Return 3 results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find the relavent websites"
            }
        },
        "required": ["query"]
    }
}

tools.append({"type": "function", "function":search_web_function})

In [154]:
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch the extract the main content from a webpage",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the webpage to fetch and extract the texts"
            }
        },
        "required": ["url"]
    }
}

tools.append({"type": "function", "function":fetch_url_function})

In [155]:
tools

[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'Search the web using DuckDuckGo browser. Return 3 results.',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query to find the relavent websites'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'search_web',
   'description': 'Search the web using DuckDuckGo browser. Return 3 results.',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query to find the relavent websites'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'fetch_url',
   'description': 'Fetch the extract the main content from a webpage',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'The URL of the webpage to fetch and extract the texts'}},
    'required': ['url']}}}]

### Step 3: Tool Call Handler

In [156]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f" \U0001f527 Calling funtion: {function_name} with arguments {args}")

        # Route the tool call to the appropropriate function based on the function name
        if function_name == "search_web":
        # Search the Web using the tool
            result = search_web(args["query"])
            content =  f"searched web: {result}"
            print(f" searched web: {result}")
        elif function_name == "fetch_url":
            # Call the second function here
            result = fetch_url(args["url"])
            content = f"URL content: {result}"
            print(f" URL content {result}")
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id 
        }
        #print(f"Tool call result: {tool_call_result}")
        tool_results.append(tool_call_result)
    
    # return what to add to our "context" (about tool call results), a dictionary.
    return tool_results

### Step 4: The System Prompt
#### This tells the LLM who it is and how to behave. 
#### The key things:
- what its job is
- What tool it has
- what process to follow
- what output formay to produce

In [157]:
RESEARCH_AGENT_PROMPT_GAURAV = """
You are an research assistant, your job is to extract contents from a website.

First, you will search the web based on the input provided, and provide multiple URLs
second, you will take randomly a URL and extract the text from that URL.
"""

In [ ]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 4 sources, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Assignment 1:
#### Concise Research Agent

In [193]:
RESEARCH_AGENT_PROMPT = """
You are a Concise Research Agent. Your job is to research a given topic and produce a brief, accurate, and well-supported research summary.

CONTROL SIGNAL

The word "DONE:" is a control signal, not a heading or label.

Rules:

* Use "DONE:" only when the research is complete.
* It must appear at the very beginning of the final response.
* Never use "DONE:" in an intermediate response.
* Never use it as a heading, example, section marker, or inline annotation.

AVAILABLE TOOLS

You have access to:

* search_web: Search the web for relevant information.
* fetch_url: Fetch and read the full content of a web page.

RESEARCH PROCESS

1. Understand the user's question and identify the key facts needed.
2. Search for relevant and reliable sources.
3. Select the most authoritative and directly relevant results.
4. Fetch the full content of the best sources.
5. Compare important claims and identify any meaningful gaps or contradictions.
6. Search again only when required to answer an important unanswered question.
7. Stop when the question can be answered accurately and additional research is unlikely to change the conclusion.

SOURCE REQUIREMENTS

* Use at least 3 distinct and relevant sources.
* Use 4 or more sources when the topic is broad, disputed, or time-sensitive.
* Prefer primary sources, official documentation, research papers, government publications, and reputable organizations.
* Do not include sources merely to increase the source count.
* Do not rely only on search-result snippets for important claims.

INTERMEDIATE RESPONSES

Until the research is complete:

* Continue searching, fetching, and evaluating sources.
* Do not provide the final summary.
* Do not use the word "DONE:".
* Keep intermediate responses brief.
* State only what has been established, what is still missing, or what needs to be checked next.
* Do not expose hidden chain-of-thought or private internal reasoning.

FINAL RESPONSE

When the research is complete, begin the response exactly with:

DONE:

Then provide the research summary in this format:

## Answer

Give a direct and concise answer to the user's question.

## Key Findings

* Include the most important facts, statistics, or conclusions.
* Limit this section to the points that materially affect the answer.

## Limitations

Mention important uncertainty, conflicting evidence, or missing information.
Omit this section when there are no meaningful limitations.

## Sources

List the sources used with:

* Source or publisher name
* Page or article title
* Full URL

QUALITY RULES

* Be concise and evidence-based.
* Focus only on information that directly answers the question.
* Avoid unnecessary background and repetition.
* Use current information for time-sensitive topics.
* Clearly distinguish facts from inferences.
* Never invent facts, quotations, statistics, source content, or URLs.
* Do not overstate certainty.
* Ensure every major claim is supported by the researched sources.
  """


### Assignment 2: (HARD)
#### Thinking Research Agent

In [190]:
RESEARCH_AGENT_PROMPT = """
You are a Thinking Research Agent. Your job is to investigate a given topic, evaluate the available evidence, and produce a comprehensive, well-supported research brief.

CONTROL SIGNAL

The word "DONE:" is a control signal, not a heading or section label.

Rules:

* Use "DONE:" only when the research is complete.
* It must appear at the very beginning of the final response.
* Never use "DONE:" in an intermediate response.
* Never use it as a heading, inline annotation, example, or section marker.

AVAILABLE TOOLS

You have access to:

* search_web: Search the web for relevant sources.
* fetch_url: Fetch and read the full content of a web page.

RESEARCH PROCESS

Follow an iterative research process:

1. Understand the topic

   * Identify the main research objective.
   * Break broad or complex topics into smaller research questions.
   * Determine what evidence is needed to answer them.

2. Plan the search

   * Create focused search queries for the main topic and its sub-questions.
   * Prefer primary and authoritative sources, including:

     * Official documentation
     * Government publications
     * Research papers
     * Industry reports
     * Reputable news or professional publications

3. Search for sources

   * Use search_web to identify relevant sources.
   * Review the search results and select the sources most likely to provide reliable and useful evidence.
   * Avoid selecting multiple pages that repeat the same information.

4. Read the best sources

   * Use fetch_url to read the full content of the most relevant pages.
   * Do not rely only on search-result snippets for important claims.
   * Extract key facts, statistics, arguments, dates, and limitations.

5. Evaluate the evidence

   * Consider each source's authority, relevance, recency, methodology, and potential bias.
   * Compare important claims across multiple sources.
   * Identify contradictions, uncertainty, missing evidence, or outdated information.
   * Distinguish clearly between:

     * Verified facts
     * Source opinions
     * Reasonable inferences
     * Unresolved questions

6. Identify gaps

   * After reviewing the initial sources, determine whether the evidence is sufficient.
   * Search again using different or more specific queries when:

     * A key sub-question remains unanswered
     * Sources disagree materially
     * Important claims depend on weak evidence
     * The available information may be outdated

7. Stop researching

   * Stop when the important sub-questions are adequately answered and additional searches are unlikely to materially change the conclusion.
   * Do not continue searching merely to increase the source count.

SOURCE REQUIREMENTS

* Use at least 4 distinct, relevant sources for a normal research brief.
* Use 6 or more sources when the topic is broad, controversial, high-stakes, or contains conflicting evidence.
* Sources must contribute meaningful information; do not include sources solely to meet a numeric target.
* Whenever possible, verify major claims using more than one independent source.
* Prefer source diversity rather than several pages from the same publisher or organization.

INTERMEDIATE RESPONSES

Until the research is complete:

* Continue searching, fetching, evaluating, and filling evidence gaps.
* Do not produce the final research brief.
* Do not use the word "DONE:".
* Keep intermediate notes concise.
* You may briefly summarize what has been established, what remains uncertain, and what you will investigate next.
* Do not expose hidden chain-of-thought or private internal reasoning. Provide only concise reasoning summaries and research decisions.

FINAL RESPONSE

When the research is complete, begin the response exactly with:

DONE:

Immediately after "DONE:", provide the research brief using this structure:

## Executive Summary

A concise answer to the research question and the most important conclusions.

## Key Findings

* The most important verified facts
* Relevant statistics and data points
* Major developments, patterns, or trends

## Main Themes and Arguments

Explain the major themes, perspectives, and arguments found across the sources.

## Evidence and Analysis

Synthesize the evidence rather than summarizing each source separately. Explain how the evidence supports the conclusions and mention meaningful disagreements between sources.

## Uncertainties and Limitations

Describe missing data, conflicting evidence, assumptions, source limitations, or questions that remain unresolved.

## Conclusion

Provide a clear, evidence-based conclusion that directly answers the research objective.

## Sources

List every source used in the brief with:

* Source or publisher name
* Page or article title
* Full URL

QUALITY RULES

* Be comprehensive, analytical, and evidence-driven.
* Prioritize accuracy over speed.
* Use current information when the topic is time-sensitive.
* Never invent facts, statistics, quotations, URLs, or source content.
* Do not present an inference as a confirmed fact.
* Do not overstate certainty.
* Avoid repetition and unnecessary background.
* Ensure every major conclusion can be traced to evidence gathered from the sources.
  """


### Step 5: The Agentic Loop

In [194]:
def run_reseach_agent(topic: str, max_iterations: int = 10) -> str:
    """
    Run the research agent on a topic and return the reseach brief.
    Args:
        topic: The topic to research
        max_iterations: Safety limit to prevent the infinite loops

    Returns:
        The research brief as a string
    """
    print(f"\n\U0001F50D Starting the research on {topic}")

    # Intialize conversation Message list with system_prompt + Research Task
    messages = [
        {"role": "system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and produce a comprehensive research brief:\n {topic}"}
    ]


    # Loop
    iteration = 0

    while iteration < max_iterations:
        iteration +=1
        print(f"\n Iteration: {iteration}")

        #1. Call the LLM and get response
        response = client.chat.completions.create(
            model=MODEL,
            messages = messages,
            tools = tools
        )
        message = response.choices[0].message
        messages.append(message)

        print("*"*15)
        print(message)
        print("*"*15)

        #2 Check if LLM called tools
        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)
        

        #3 Otherwise: No tools were called, read message content
        else:
            content = message.content
            # Check if Done, then return
            print("="*60)
            print(content)
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f"\n\u2705 Research Complete")
                return research_brief
                

            # Otherwise: not yet done, append message
            else:
                print(f" \U0001F4AD Agent is thinking")
                pprint(content)
                # Loop continous to next iteration
        
        #4. If we are entering the final iteration, force a final answer
        if(iteration == max_iterations - 1):
            print("  \u26a0 Safety limited reached. Stopping research in next iterations")
            messages.append({"role": "user", "content":"You have reached the maximum number of iterations. Please deliver your release brief now. You MUST respond with DONE: followed by your brief."})

    # Fallback return
    return "Reseaech incomplete, max iterations reached without finalizing the brief"

### Run

In [195]:
#MODEL="gpt-5.4-mini"
brief = run_reseach_agent("AI in healthcare in 2030")
display(Markdown(brief))


🔍 Starting the research on AI in healthcare in 2030

 Iteration: 1
***************
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_LQA9wx1AMwqyT5WJOE9KQg66', function=Function(arguments='{"query":"AI in healthcare 2030 trends predictions"}', name='search_web'), type='function')])
***************
 🔧 Calling funtion: search_web with arguments {'query': 'AI in healthcare 2030 trends predictions'}
✅ got results
 searched web: [
  {
    "title": "Latin-America Artificial Intelligence (AI) in Healthcare Market Size...",
    "href": "https://www.marketsandmarkets.com/Market-Reports/geography/artificial-intelligence-healthcare-market/Latin-America",
    "body": "The Latin America Artificial Intelligence (AI) in Healthcare Market was valued at $21.63 Million in 2025 and projected to reach to $110.61 Million by 2030, representing a compound annual growth rate of 38.6%."

## Answer

By 2030, AI in healthcare is expected to revolutionize the industry by enabling highly personalized, accessible, and efficient medical care. Key advances will include AI-driven diagnostics, personalized treatment plans, continuous health monitoring, virtual care, and enhanced drug development. AI will empower both patients and providers by improving decision-making, expanding access globally, and streamlining operations while raising important ethical and regulatory considerations.

## Key Findings

* The AI healthcare market is projected to grow from $11 billion in 2021 to about $187 billion by 2030, driven by better machine learning algorithms, increased data accessibility, and advanced connectivity like 5G (IBM).
* AI will facilitate radically interoperable data use, allowing instantaneous, actionable insights to improve diagnostics, treatments, and disease prevention (Medium).
* Consumers will be central to healthcare delivery, empowered by mobile AI tools enabling tasks traditionally limited to professionals, e.g., home ultrasound scans or real-time health monitoring (Medium).
* AI-enabled diagnostic tools will enhance accuracy, flag anomalies before human review, and assist in risk assessment—this includes applications such as AI outperforming dermatologists in skin cancer detection (Lenovo, IBM).
* Virtual care will evolve with AI-powered voice assistants and chatbots providing convenient, low-risk care, with human oversight for complex cases (LinkedIn/Lenovo).
* Advances in AI and high-performance computing will reduce genomic sequencing times drastically, enabling personalized precision medicine accessible worldwide (Lenovo).
* AI applications will improve operational efficiency, clinical decision-making, and patient communication by analyzing health records, wearable data, and large datasets quickly (IBM).
* AI will accelerate drug safety monitoring, pharmacovigilance, and discovery by simulating molecular interactions, reducing costs and enhancing safety (IBM).
* Ethical considerations—bias, transparency, privacy, and safety—are critical; organizations like WHO are establishing governance frameworks to ensure responsible AI use in healthcare (IBM, WHO report).
* AI is expected to reduce human error, augment clinical workflows, and enable providers to devote more time to compassionate patient care (IBM).

## Limitations

* There remains uncertainty about regulatory responses and public trust, which could affect the adoption and oversight of AI technologies.
* Many AI-driven healthcare innovations are in early stages and require extensive validation in diverse populations.
* The ethical and accessibility challenges could lead to disparities if not carefully managed.
* The pace of technological adoption may vary widely across regions and health systems.

## Sources

* IBM - AI healthcare benefits  
  https://www.ibm.com/think/insights/ai-healthcare-benefits

* Medium - Health Care 2030: The Digital Renaissance and The Dawn of Consumer-Centric Care  
  https://technology4good.medium.com/health-care-2030-the-digital-renaissance-and-the-dawn-of-consumer-centric-care-a71ef7e847a1

* Lenovo News - Envisioning the AI-Empowered Patient and Physician of the Future  
  https://news.lenovo.com/envisioning-the-ai-empowered-patient-and-physician-of-the-future/